# Anonimisasi Foto Contoh (Blur Pelat Nomor dan Wajah, EXIF Dipertahankan)

Notebook ini menyiapkan foto contoh untuk repo publik (`sample_data/photos/`):

1. **Menyamarkan informasi pribadi**: pelat nomor kendaraan dan kepala/wajah pengendara.
2. **Mempertahankan EXIF** (GPS, heading kompas, model kamera, tanggal, focal length), karena pipeline geolokasi (`03`) membutuhkan semuanya.
3. **Membuang thumbnail EXIF.** Kamera ponsel menyimpan salinan kecil foto asli di dalam EXIF (sekitar 10 KB). Kalau EXIF disalin apa adanya, pelat nomor masih terlihat di thumbnail itu walaupun foto utamanya sudah di-blur.

**Kenapa tidak di aplikasi edit foto (Paint, dll.):** kebanyakan aplikasi membuang EXIF saat menyimpan. Di sini EXIF dibaca dari file asli dengan `piexif` lalu ditulis kembali ke file hasil.

**Cara kerja:**
1. Tampilkan foto dengan grid koordinat piksel → tentukan kotak area yang mau disamarkan.
2. Zoom ke area tersebut untuk memastikan kotaknya tepat.
3. Pixelate (perkecil 10×, perbesar lagi) lalu Gaussian blur di dalam kotak. Pixelate membuang detail sehingga tulisan pelat tidak bisa direkonstruksi; blur kecil sesudahnya hanya untuk merapikan tampilan.
4. Simpan dengan EXIF asli minus thumbnail.
5. Verifikasi: zoom ke area yang disamarkan dan cek isi EXIF.

**Foto asli tidak diubah.** Hasil ditulis ke folder terpisah.

## 0. Setup

In [ ]:
!pip install -q piexif exifread

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import piexif
import exifread
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFilter

SRC_DIR = Path("/content/drive/MyDrive/sidewalk_photos/")             # foto asli (tidak diubah)
DST_DIR = Path("/content/drive/MyDrive/sidewalk_photos_anonymized/")  # hasil untuk repo
DST_DIR.mkdir(exist_ok=True)

PHOTO = "IMG_20250905_172550.jpg"   # foto contoh: area terbuka, hasil geolokasi terbaik

## 1. Grid koordinat untuk menentukan kotak

Garis kuning setiap 250 piksel, label koordinat di tepi. Koordinat ini dalam piksel **foto asli** (4160 × 3120 untuk OPPO A7), bukan piksel tampilan.

In [ ]:
def show_grid(path, step=250, figsize=(18, 13.5)):
    img = Image.open(path).convert("RGB")
    d = ImageDraw.Draw(img)
    for x in range(0, img.width, step):
        d.line([(x, 0), (x, img.height)], fill=(255, 255, 0), width=3)
    for y in range(0, img.height, step):
        d.line([(0, y), (img.width, y)], fill=(255, 255, 0), width=3)
    plt.figure(figsize=figsize); plt.imshow(img)
    plt.xticks(range(0, img.width, step), rotation=90); plt.yticks(range(0, img.height, step))
    plt.title(path.name); plt.show()

show_grid(SRC_DIR / PHOTO)

## 2. Zoom ke area yang berisi informasi pribadi

Untuk foto ini, semua pelat dan pengendara ada di badan jalan di sekitar tengah gambar (x 1400–3200, y 1300–1800), plus satu motor parkir di kiri (x 500–1100, y 1500–2100). Zoom dipakai untuk membaca posisi pelat dengan teliti: posisi di crop + offset crop = posisi di foto asli.

In [ ]:
ZOOMS = {
    "jalan (tengah)": (1400, 1300, 3200, 1800),
    "motor parkir (kiri)": (500, 1500, 1100, 2100),
}

def show_zooms(path, zooms):
    img = Image.open(path)
    for title, (x1, y1, x2, y2) in zooms.items():
        plt.figure(figsize=(16, 16 * (y2 - y1) / (x2 - x1)))
        plt.imshow(img.crop((x1, y1, x2, y2)), extent=(x1, x2, y2, y1))   # sumbu = koordinat foto asli
        plt.title(title); plt.grid(True, color="yellow", alpha=0.5); plt.show()

show_zooms(SRC_DIR / PHOTO, ZOOMS)

## 3. Kotak area yang disamarkan

Kotak `(x1, y1, x2, y2)` dalam piksel foto asli, sengaja diberi margin beberapa piksel di sekeliling pelat/kepala supaya tepi tulisan tidak tersisa.

Yang disamarkan untuk `IMG_20250905_172550.jpg`:
- **Pelat nomor (4):** Avanza hitam parkir, Innova hitam, motor pengendara berbaju cokelat, motor merah parkir di kiri.
- **Kepala pengendara (3):** pengendara berbaju cokelat, pengendara skuter hijau, pengendara berhelm putih.
- **Mobil putih di tepi kiri:** area belakang, untuk berjaga-jaga kalau pelatnya sebagian terlihat.

Pengendara dan kendaraan yang jauh terlalu kecil untuk dikenali, jadi tidak disamarkan. Semua kotak berada di badan jalan, jauh dari trotoar dan obstacle (bollard, guiding block), sehingga hasil pipeline geolokasi untuk foto ini tidak berubah.

In [ ]:
BLUR_BOXES = {
    "IMG_20250905_172550.jpg": {
        "pelat Avanza hitam (parkir)":   (1805, 1578, 1940, 1638),
        "pelat Innova hitam":            (2255, 1512, 2330, 1552),
        "pelat motor pengendara cokelat": (2430, 1625, 2505, 1672),
        "pelat motor merah (parkir)":    (730, 1715, 850, 1775),
        "kepala pengendara cokelat":     (2390, 1475, 2465, 1545),
        "kepala pengendara skuter hijau": (3050, 1470, 3115, 1530),
        "kepala pengendara helm putih":  (2040, 1485, 2100, 1545),
        "belakang mobil putih (kiri)":   (1405, 1540, 1500, 1600),
    },
}

In [ ]:
# Pratinjau kotak sebelum diproses (merah = area yang akan disamarkan)
def preview_boxes(path, boxes):
    img = Image.open(path).convert("RGB")
    d = ImageDraw.Draw(img)
    for (x1, y1, x2, y2) in boxes.values():
        d.rectangle([x1, y1, x2, y2], outline=(255, 0, 0), width=6)
    plt.figure(figsize=(16, 16 * 500 / 1800))
    plt.imshow(img.crop((500, 1300, 3200, 1800)), extent=(500, 3200, 1800, 1300))
    plt.title("Pratinjau kotak"); plt.show()

preview_boxes(SRC_DIR / PHOTO, BLUR_BOXES[PHOTO])

## 4. Samarkan dan simpan dengan EXIF asli

**Pixelate lalu blur:**
- `resize` ke 1/10 ukuran lalu kembali ke ukuran semula dengan `NEAREST` → detail hilang permanen (blok 10×10 piksel). Pada pelat setinggi ~50 piksel, huruf tidak lagi bisa dibaca maupun direkonstruksi.
- `GaussianBlur(4)` sesudahnya hanya merapikan tepi blok supaya tampilan tidak terlalu kasar.

**EXIF:**
- `piexif.load` membaca seluruh EXIF dari file asli (GPS, heading, Make/Model, DateTimeOriginal, FocalLength).
- `exif["thumbnail"] = None` dan `exif["1st"] = {}` membuang thumbnail EXIF (IFD1), yang berisi foto asli tanpa sensor.
- `quality=95` menjaga kualitas JPEG tetap tinggi, supaya hasil SegFormer/YOLO/Depth Pro tidak terpengaruh kompresi.

In [ ]:
def anonymize(src, dst, boxes, pixel=10, blur_radius=4, quality=95):
    img = Image.open(src)
    exif = piexif.load(img.info["exif"])            # EXIF asli lengkap
    for (x1, y1, x2, y2) in boxes.values():
        region = img.crop((x1, y1, x2, y2))
        w, h = region.size
        region = (region.resize((max(1, w // pixel), max(1, h // pixel)), Image.BILINEAR)   # buang detail
                        .resize((w, h), Image.NEAREST)                                      # blok piksel
                        .filter(ImageFilter.GaussianBlur(blur_radius)))                     # rapikan tepi
        img.paste(region, (x1, y1))
    exif["thumbnail"] = None   # thumbnail EXIF = foto asli tanpa sensor -> buang
    exif["1st"] = {}           # IFD1 (metadata thumbnail) ikut dikosongkan
    img.save(dst, quality=quality, exif=piexif.dump(exif))


for name, boxes in BLUR_BOXES.items():
    anonymize(SRC_DIR / name, DST_DIR / name, boxes)
    print("ok:", DST_DIR / name)

## 5. Verifikasi

1. **Visual:** zoom ke area yang sama seperti Bagian 2; semua pelat dan kepala dalam kotak harus tidak terbaca.
2. **EXIF:** tag yang dipakai pipeline harus tetap ada, dan `thumbnail` harus `False`.

In [ ]:
show_zooms(DST_DIR / PHOTO, ZOOMS)

In [ ]:
TAGS = ["GPS GPSLatitude", "GPS GPSLongitude", "GPS GPSImgDirection", "GPS GPSImgDirectionRef",
        "Image Make", "Image Model", "EXIF DateTimeOriginal", "EXIF FocalLength"]

for name in BLUR_BOXES:
    with open(SRC_DIR / name, "rb") as f:
        t_src = exifread.process_file(f, details=False)
    with open(DST_DIR / name, "rb") as f:
        t_dst = exifread.process_file(f, details=False)
    print(f"== {name}")
    for k in TAGS:
        same = str(t_src.get(k)) == str(t_dst.get(k))
        print(f"  {k:28s} {str(t_dst.get(k)):28s} {'sama' if same else 'BERBEDA!'}")
    print(f"  thumbnail asli: {'JPEGThumbnail' in t_src}  ->  hasil: {'JPEGThumbnail' in t_dst}  (harus False)")

## Catatan

- Untuk foto lain, tambahkan entri baru di `BLUR_BOXES` dan ulangi Bagian 1–5.
- Jangan jalankan anonimisasi dua kali pada file hasil: EXIF tetap aman, tapi kualitas JPEG turun karena kompresi berulang. Selalu proses dari foto asli.
- Hasil di `sidewalk_photos_anonymized/` inilah yang disalin ke `sample_data/photos/` di repo.